# 05 · Linear Regression
**Owner:** Senanayake · **Branch:** `model/linear-regression`

GroupKFold CV on the training set (split by battery) + fit on full train + predict on the held-out test set. Metrics saved for notebook 10.


In [23]:
# --- Colab setup (run first) ---
import os
REPO = '/content/ev-battery-rul-prediction'   # absolute path -> safe to re-run
if not os.path.exists(REPO):
    !git clone https://github.com/Chathumi-Senanayake/ev-battery-rul-prediction.git {REPO}
%cd {REPO}
!git pull -q
!pip install -q -r requirements.txt
import sys
if REPO not in sys.path: sys.path.insert(0, REPO)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/ev_battery_data'
except Exception as e:
    print('Drive mount failed -> using local storage:', e)
    DATA_DIR = '/content/ev_battery_data'
os.makedirs(DATA_DIR, exist_ok=True)
print('DATA_DIR =', DATA_DIR)


/content/ev-battery-rul-prediction
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_DIR = /content/drive/MyDrive/ev_battery_data


In [24]:
import os, json, numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score
train = pd.read_parquet(os.path.join(DATA_DIR,'train.parquet'))
test  = pd.read_parquet(os.path.join(DATA_DIR,'test.parquet'))
feats = json.load(open(os.path.join(DATA_DIR,'selected_features.json')))
Xtr, ytr, gtr = train[feats], train['RUL'], train['cell']
Xte, yte = test[feats], test['RUL']


In [25]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LinearRegression
model = make_pipeline(RobustScaler(), LinearRegression())


In [26]:

gkf = GroupKFold(n_splits=5); maes, r2s = [], []
for tr, va in gkf.split(Xtr, ytr, gtr):
    model.fit(Xtr.iloc[tr], ytr.iloc[tr])
    p = model.predict(Xtr.iloc[va])
    maes.append(mean_absolute_error(ytr.iloc[va], p)); r2s.append(r2_score(ytr.iloc[va], p))
print(f'CV  MAE {np.mean(maes):.1f}+/-{np.std(maes):.1f}  R2 {np.mean(r2s):.3f}')


CV  MAE 142.5+/-17.8  R2 0.722


In [27]:

model.fit(Xtr, ytr); pte = model.predict(Xte)
test_mae = mean_absolute_error(yte, pte); test_r2 = r2_score(yte, pte)
print(f'TEST MAE {test_mae:.1f}  R2 {test_r2:.3f}')


TEST MAE 221.7  R2 -0.491


In [28]:
metrics = {'model':'Linear Regression','member':'Senanayake',
   'cv_MAE':float(np.mean(maes)),'cv_R2':float(np.mean(r2s)),
   'test_MAE':float(test_mae),'test_R2':float(test_r2)}
json.dump(metrics, open(os.path.join(DATA_DIR,'metrics_05.json'),'w'), indent=2)
pd.DataFrame({'cell':test['cell'],'RUL':yte,'pred':pte}).to_csv(os.path.join(DATA_DIR,'pred_05.csv'), index=False)
metrics


{'model': 'Linear Regression',
 'member': 'Senanayake',
 'cv_MAE': 142.51019344672036,
 'cv_R2': 0.721908572083152,
 'test_MAE': 221.71472082004027,
 'test_R2': -0.4907669670773043}